In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2009_2010"
    / "Labs"
)

xpt_file = DATA_DIR / "FASTQX_F.xpt"
csv_file = DATA_DIR / "FASTQX_F_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve untouched source data.
#
source_df = df.copy()


print("=" * 75)
print("NHANES 2009-2010 FASTQX_F XPT -> CSV")
print("=" * 75)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "PHQ020",
    "PHACOFHR",
    "PHACOFMN",
    "PHQ030",
    "PHAALCHR",
    "PHAALCMN",
    "PHQ040",
    "PHAGUMHR",
    "PHAGUMMN",
    "PHQ050",
    "PHAANTHR",
    "PHAANTMN",
    "PHQ060",
    "PHASUPHR",
    "PHASUPMN",
    "PHAFSTHR",
    "PHAFSTMN",
    "PHDSESN"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "FASTQX_F column names/order differ from expected.\n"
        f"Actual: {df.columns.tolist()}"
    )


if len(df) != 9835:

    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )


if len(df.columns) != 19:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 9,835 rows x 19 columns."
)


# ============================================================
# 4. CHECK FOR KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known SAS XPORT / pandas zero-decoding artifact:
#
#     5.397605346934028e-79
#
# IMPORTANT:
#
# Never globally replace this value before validating
# exactly where it occurs and confirming zero is valid.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_total = int(
    df[numeric_cols]
    .eq(0)
    .sum()
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "True numeric zeros:",
    f"{true_zero_total:,}"
)


print(
    "Tiny artifact occurrences:",
    f"{tiny_total:,}"
)


if tiny_total > 0:

    print(
        "\nTiny values by variable:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


# ============================================================
# 5. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Validated against:
#
# 1. The attached FASTQX_F.xpt
# 2. CDC 2009-2010 FASTQX_F codebook
#
# CDC confirms zero is valid in these variables.
#

EXPECTED_TINY_COUNTS = {

    "PHDSESN": 4589,

    "PHAFSTHR": 893,

    "PHAFSTMN": 177,

    "PHAGUMHR": 46,

    "PHAGUMMN": 1,

    "PHAANTMN": 1,

    "PHASUPHR": 1
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated FASTQX_F pattern.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {actual_tiny_counts}"
    )


if tiny_total != 5708:

    raise ValueError(
        f"Unexpected total tiny-value count: {tiny_total}"
    )


print(
    "\nPASS: Tiny-value pattern matches "
    "validated FASTQX_F pattern."
)


# ============================================================
# 6. RESTORE VALIDATED ZERO VALUES
# ============================================================

#
# Restore ONLY the exact validated tiny values.
#
# No blanket global replacement.
#

print("\n--- ZERO RESTORATION ---")


correction_total = 0


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    tiny_mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        tiny_mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: unexpected tiny-value count."
        )


    df.loc[
        tiny_mask,
        col
    ] = 0


    correction_total += actual_count


    print(
        f"{col}: "
        f"{actual_count:,} restored to 0"
    )


if correction_total != 5708:

    raise ValueError(
        "Incorrect total zero-restoration count."
    )


print(
    "\nPASS:",
    f"{correction_total:,}",
    "validated zero values restored."
)


# ============================================================
# 7. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df represents exactly what the corrected
# dataset should contain.
#
# It differs from source_df ONLY at validated
# tiny-zero positions.
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col].eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 8. CHECK FOR GENUINE DECIMAL VALUES
# ============================================================

#
# CRITICAL FIDELITY RULE:
#
# Never round genuine decimals.
#
# This check is always performed BEFORE Int64 conversion.
#
# For FASTQX_F, no genuine fractional values exist.
#
# If any are discovered, STOP instead of rounding.
#
# This same rule protects Dietary, Laboratory,
# Examination, and other continuous measurements.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if fractional_mask.any():

        fractional_columns.append(
            col
        )

        print(
            f"\nGenuine decimal values in {col}:"
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal values detected. "
        "They will NOT be rounded or changed:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional values "
    "exist in FASTQX_F."
)


# ============================================================
# 9. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


if (
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
    or
    df["SEQN"].min()
    != 51624
    or
    df["SEQN"].max()
    != 62160
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 10. FORMAT WHOLE-NUMBER VARIABLES WITHOUT .0
# ============================================================

#
# All FASTQX_F fields contain whole-number values
# after validated zero restoration.
#
# Hours and minutes are integer values.
#
# NO .round() is used.
#

for col in df.columns:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_number_check = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not np.all(
            whole_number_check
        ):

            raise ValueError(
                f"{col} contains genuine decimal values "
                "and cannot safely be converted to Int64."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All FASTQX_F fields formatted "
    "without unnecessary .0."
)


# ============================================================
# 11. CDC CHECK - YES / NO QUESTIONS
# ============================================================

#
# CDC published frequencies:
#
# PHQ020
#   Yes = 21
#   No  = 9537
#   Missing = 277
#
# PHQ030
#   Yes = 6
#   No  = 9552
#   Missing = 277
#
# PHQ040
#   Yes = 79
#   No  = 9479
#   Missing = 277
#
# PHQ050
#   Yes = 6
#   No  = 9552
#   Missing = 277
#
# PHQ060
#   Yes = 17
#   No  = 9541
#   Missing = 277
#

CDC_QUESTION_COUNTS = {

    "PHQ020": {
        "values": {
            1: 21,
            2: 9537
        },
        "missing": 277
    },

    "PHQ030": {
        "values": {
            1: 6,
            2: 9552
        },
        "missing": 277
    },

    "PHQ040": {
        "values": {
            1: 79,
            2: 9479
        },
        "missing": 277
    },

    "PHQ050": {
        "values": {
            1: 6,
            2: 9552
        },
        "missing": 277
    },

    "PHQ060": {
        "values": {
            1: 17,
            2: 9541
        },
        "missing": 277
    }
}


print("\n--- CDC QUESTION CHECKS ---")


for col, expected in CDC_QUESTION_COUNTS.items():

    actual_values = (
        df[col]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[col]
        .isna()
        .sum()
    )


    if actual_values != expected["values"]:

        raise ValueError(
            f"{col} frequencies differ from CDC.\n"
            f"Expected: {expected['values']}\n"
            f"Actual:   {actual_values}"
        )


    if actual_missing != expected["missing"]:

        raise ValueError(
            f"{col} missing count differs from CDC.\n"
            f"Expected: {expected['missing']}\n"
            f"Actual:   {actual_missing}"
        )


    print(
        f"PASS: {col} matches CDC."
    )


# ============================================================
# 12. CDC CHECK - FASTING TIME VARIABLES
# ============================================================

#
# Format:
#
# variable:
# (
#     minimum,
#     maximum,
#     expected non-missing,
#     expected missing
# )
#

CDC_RANGE_CHECKS = {

    "PHACOFHR": (
        1,
        19,
        21,
        9814
    ),

    "PHACOFMN": (
        1,
        54,
        21,
        9814
    ),

    "PHAALCHR": (
        1,
        15,
        6,
        9829
    ),

    "PHAALCMN": (
        10,
        56,
        6,
        9829
    ),

    "PHAGUMHR": (
        0,
        16,
        79,
        9756
    ),

    "PHAGUMMN": (
        0,
        59,
        79,
        9756
    ),

    "PHAANTHR": (
        1,
        9,
        6,
        9829
    ),

    "PHAANTMN": (
        0,
        54,
        6,
        9829
    ),

    "PHASUPHR": (
        0,
        24,
        17,
        9818
    ),

    "PHASUPMN": (
        4,
        58,
        17,
        9818
    ),

    "PHAFSTHR": (
        0,
        39,
        9557,
        278
    ),

    "PHAFSTMN": (
        0,
        59,
        9557,
        278
    )
}


print("\n--- CDC FASTING TIME CHECKS ---")


for col, (
    expected_min,
    expected_max,
    expected_nonmissing,
    expected_missing
) in CDC_RANGE_CHECKS.items():

    s = df[col]


    actual_min = s.min()
    actual_max = s.max()


    actual_nonmissing = int(
        s.notna()
        .sum()
    )


    actual_missing = int(
        s.isna()
        .sum()
    )


    if actual_min != expected_min:

        raise ValueError(
            f"{col} minimum differs from CDC.\n"
            f"Expected: {expected_min}\n"
            f"Actual: {actual_min}"
        )


    if actual_max != expected_max:

        raise ValueError(
            f"{col} maximum differs from CDC.\n"
            f"Expected: {expected_max}\n"
            f"Actual: {actual_max}"
        )


    if actual_nonmissing != expected_nonmissing:

        raise ValueError(
            f"{col} non-missing count differs from CDC.\n"
            f"Expected: {expected_nonmissing}\n"
            f"Actual: {actual_nonmissing}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{col} missing count differs from CDC.\n"
            f"Expected: {expected_missing}\n"
            f"Actual: {actual_missing}"
        )


    print(
        f"PASS: {col} matches CDC."
    )


# ============================================================
# 13. VERIFY VALIDATED ZERO VALUES
# ============================================================

#
# These zeros are particularly important because
# they were decoded as tiny XPORT values.
#

ZERO_COUNTS = {

    "PHDSESN": 4589,

    "PHAFSTHR": 893,

    "PHAFSTMN": 177,

    "PHAGUMHR": 46,

    "PHAGUMMN": 1,

    "PHAANTMN": 1,

    "PHASUPHR": 1
}


print("\n--- ZERO RESTORATION CHECK ---")


for col, expected_zero_count in ZERO_COUNTS.items():

    actual_zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    print(
        f"{col} zeros:",
        actual_zero_count
    )


    if actual_zero_count != expected_zero_count:

        raise ValueError(
            f"{col} zero restoration failed.\n"
            f"Expected: {expected_zero_count}\n"
            f"Actual: {actual_zero_count}"
        )


print(
    "PASS: All validated zero counts are correct."
)


# ============================================================
# 14. CDC CHECK - PHDSESN
# ============================================================

#
# CDC:
#
# 0 = Morning   = 4,589
# 1 = Afternoon = 3,471
# 2 = Evening   = 1,775
#
# Missing = 0
#
# The 4,589 raw tiny values therefore represent
# the CDC code 0 = morning.
#

EXPECTED_PHDSESN = {
    0: 4589,
    1: 3471,
    2: 1775
}


actual = (
    df["PHDSESN"]
    .value_counts()
    .sort_index()
    .to_dict()
)


print("\n--- PHDSESN ---")


print(
    actual
)


if (
    actual != EXPECTED_PHDSESN
    or
    int(
        df["PHDSESN"]
        .isna()
        .sum()
    )
    != 0
):

    raise ValueError(
        "PHDSESN differs from CDC."
    )


print(
    "PASS: PHDSESN matches CDC."
)


# ============================================================
# 15. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# Expected differences from raw pandas XPT:
#
# TINY_VALUE -> 0
#
# ONLY in:
#
# PHDSESN
# PHAFSTHR
# PHAFSTMN
# PHAGUMHR
# PHAGUMMN
# PHAANTMN
# PHASUPHR
#
# NOTHING ELSE may change numerically.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        failures.append(
            col
        )


if failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(failures)
    )


print(
    "PASS: All 19 variables match expected "
    "corrected XPT values exactly."
)


# ============================================================
# 16. EXPORT CSV
# ============================================================

#
# IMPORTANT:
#
# NO float_format.
# NO rounding.
#
# Missing values become blank.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 17. RE-READ EXPORTED CSV
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 18. EXACT CORRECTED XPT -> CSV FIDELITY
# ============================================================

print("\n--- XPT -> CSV FIDELITY ---")


csv_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV differences found in:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 19 CSV columns match "
    "expected corrected XPT values."
)


# ============================================================
# 19. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in expected_df.columns:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE CHECK ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 20. CHECK CSV FOR REMAINING TINY VALUES
# ============================================================

tiny_csv_count = 0


for col in csv_df.select_dtypes(
    include=[np.number]
).columns:

    tiny_csv_count += int(
        csv_df[col]
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in CSV."
    )


print(
    "PASS: No tiny-value artifacts remain."
)


# ============================================================
# 21. RAW CSV .0 FORMATTING CHECK
# ============================================================

#
# FASTQX_F has NO genuine decimal columns.
#
# Therefore ".0" should not appear anywhere
# in the final CSV.
#

print("\n--- RAW CSV FORMAT CHECK ---")


with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


dot_zero_count = (
    csv_text.count(".0")
)


print(
    "Occurrences of '.0':",
    dot_zero_count
)


if dot_zero_count != 0:

    raise ValueError(
        "Unexpected .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 formatting."
)


# ============================================================
# 22. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 23. FINAL CSV SEQN CHECK
# ============================================================

print("\n--- FINAL CSV SEQN CHECK ---")


if (
    csv_df["SEQN"].min()
    != 51624
    or
    csv_df["SEQN"].max()
    != 62160
    or
    int(
        csv_df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        csv_df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


print(
    "PASS: Final CSV SEQN is valid."
)


# ============================================================
# 24. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 75
)


print(
    "FASTQX_F CONVERSION COMPLETE"
)


print(
    "=" * 75
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "Validated zero restorations:",
    f"{correction_total:,}"
)


print(
    "Genuine decimal variables: NONE"
)


print(
    "XPT -> CSV unexpected differences: 0 expected"
)


print(
    "Missing-value differences: 0 expected"
)


print(
    "Tiny values remaining: 0 expected"
)


print(
    "Unnecessary .0: 0 expected"
)

NHANES 2009-2010 FASTQX_F XPT -> CSV
Rows: 9,835
Columns: 19
PASS: Structure = 9,835 rows x 19 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 5,708

Tiny values by variable:
PHDSESN     4589
PHAFSTHR     893
PHAFSTMN     177
PHAGUMHR      46
PHAGUMMN       1
PHAANTMN       1
PHASUPHR       1

PASS: Tiny-value pattern matches validated FASTQX_F pattern.

--- ZERO RESTORATION ---
PHDSESN: 4,589 restored to 0
PHAFSTHR: 893 restored to 0
PHAFSTMN: 177 restored to 0
PHAGUMHR: 46 restored to 0
PHAGUMMN: 1 restored to 0
PHAANTMN: 1 restored to 0
PHASUPHR: 1 restored to 0

PASS: 5,708 validated zero values restored.

--- FRACTIONAL VALUE CHECK ---
PASS: No genuine fractional values exist in FASTQX_F.

--- SEQN ---
Missing: 0
Duplicates: 0
Minimum: 51624
Maximum: 62160
PASS: SEQN validated.

PASS: All FASTQX_F fields formatted without unnecessary .0.

--- CDC QUESTION CHECKS ---
PASS: PHQ020 matches CDC.
PASS: PHQ030 matches CDC.
PASS: PHQ040 matches 